# Findie · thử Qwen3-VL-Embedding-2B trên Colab Free

Chọn **Runtime → Change runtime type → T4 GPU**, sau đó chạy các cell theo thứ tự.
Notebook tự tạo dữ liệu giả lập; tùy chọn upload chỉ dành cho ảnh giả lập/không nhạy cảm.
Không đưa giấy tờ, khuôn mặt, tin nhắn, đáp án xác minh hoặc credential thật lên Colab.
Không mount Drive, không kết nối DB và không mở FastAPI/tunnel. GPU Free có thể không được cấp,
hết quota hoặc bị ngắt. Nếu lỗi thiếu bộ nhớ, dùng **Disconnect and delete runtime** rồi chạy lại
một notebook; không nạp hai model cùng lúc. Đây là thử nghiệm inference, không phải huấn luyện.

Chạy văn bản → văn bản và ảnh → ảnh riêng, vector **2048 chiều, L2**, xếp hạng cosine.
Điểm tương đồng không phải xác suất sở hữu; ví dụ ảnh tự sinh chỉ kiểm tra luồng chạy.
Ghim trọng số/processor/mã official và ghi manifest; chưa đo chất lượng tiếng Việt hoặc VRAM thực tế.
[Model card chính thức](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B)
· [Mã inference được ghim](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B/blob/9f2f7e710d6d81056aa5c0a4f04764fec6bb7bda/scripts/qwen3_vl_embedding.py).

In [ ]:
# Giữ nguyên bộ PyTorch/CUDA do Colab cung cấp; không cài torch hoặc FlashAttention.
import importlib.metadata as metadata
import pathlib
import subprocess
import sys
import tempfile

if not (3, 10) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Notebook cần Python 3.10–3.13; chọn runtime Colab tương thích.")
PINS = {'transformers': '4.57.3', 'accelerate': '1.12.0', 'qwen-vl-utils': '0.0.14', 'huggingface-hub': '0.36.0', 'numpy': '2.2.6', 'pillow': '11.3.0'}
runtime_torch = {}
for name in ("torch", "torchvision", "torchaudio"):
    try:
        runtime_torch[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        if name != "torchaudio":
            raise RuntimeError("Chọn runtime GPU Colab có sẵn torch và torchvision.")
with tempfile.TemporaryDirectory() as directory:
    constraint = pathlib.Path(directory) / "colab-torch.txt"
    constraint.write_text("\n".join(f"{name}=={version}" for name, version in runtime_torch.items()))
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
        "--upgrade-strategy", "only-if-needed", "--constraint", str(constraint),
        *(f"{name}=={version}" for name, version in PINS.items()),
    ], check=True)
assert all(metadata.version(name) == version for name, version in runtime_torch.items())
print("Đã cài gói ghim phiên bản và giữ nguyên:", runtime_torch)
print("Nếu Colab báo gói đã được import: Runtime > Restart session, rồi chạy từ cell này.")

In [ ]:
import gc
import hashlib
import io
import json
import math
import platform
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torchvision
from packaging.version import Version
from PIL import Image, ImageDraw, ImageOps
from IPython.display import display

for name, expected in PINS.items():
    if metadata.version(name) != expected:
        raise RuntimeError(f"Sai phiên bản {name}; khởi động lại session và chạy setup.")
if Version(torch.__version__.split("+")[0]) < Version("2.8.0"):
    raise RuntimeError("Cần runtime Colab với PyTorch >= 2.8; không thay bộ CUDA trong notebook.")
if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU: Runtime > Change runtime type > T4 GPU. Không chạy CPU.")
torch.cuda.empty_cache()
free_bytes, total_bytes = torch.cuda.mem_get_info()
if free_bytes < 7 * 1024**3:
    raise RuntimeError("Cần ít nhất 7 GiB VRAM trống. Disconnect and delete runtime rồi chọn GPU lại.")
torch.manual_seed(42)
np.random.seed(42)
ENVIRONMENT = {
    "python": platform.python_version(), "torch": torch.__version__,
    "torchvision": torchvision.__version__, "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0), "gpu_total_bytes": total_bytes,
    "packages": dict(sorted((dist.metadata["Name"], dist.version)
                            for dist in metadata.distributions() if dist.metadata["Name"])),
}
print({key: value for key, value in ENVIRONMENT.items() if key != "packages"})

In [ ]:
MAX_UPLOAD_BYTES = 10 * 1024 * 1024
MAX_SOURCE_PIXELS = 20_000_000
MAX_IMAGE_EDGE = 1024
Image.MAX_IMAGE_PIXELS = MAX_SOURCE_PIXELS

def decode_image(raw):
    if not isinstance(raw, bytes) or not 0 < len(raw) <= MAX_UPLOAD_BYTES:
        raise ValueError("Ảnh phải là bytes, tối đa 10 MiB.")
    with warnings.catch_warnings():
        warnings.simplefilter("error", Image.DecompressionBombWarning)
        with Image.open(io.BytesIO(raw)) as candidate:
            if candidate.format not in {"JPEG", "PNG", "WEBP"}:
                raise ValueError("Chỉ nhận nội dung JPEG/PNG/WebP.")
            if getattr(candidate, "n_frames", 1) != 1:
                raise ValueError("Chỉ nhận ảnh tĩnh một frame.")
            if not 0 < candidate.width * candidate.height <= MAX_SOURCE_PIXELS:
                raise ValueError("Kích thước ảnh giải mã vượt giới hạn 20 megapixel.")
            candidate.verify()
        with Image.open(io.BytesIO(raw)) as candidate:
            oriented = ImageOps.exif_transpose(candidate).convert("RGB")
            oriented.thumbnail((MAX_IMAGE_EDGE, MAX_IMAGE_EDGE), Image.Resampling.LANCZOS)
            return Image.frombytes("RGB", oriented.size, oriented.tobytes())

def image_sha256(image):
    dimensions = f"RGB:{image.width}:{image.height}:".encode()
    return hashlib.sha256(dimensions + image.tobytes()).hexdigest()

def upload_one_synthetic_image():
    from google.colab import files
    # files.upload ghi vào runtime; dọn tệp được tạo ngay cả khi giải mã lỗi.
    before = {path.resolve() for path in Path.cwd().iterdir()}
    uploaded = {}
    try:
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Mỗi lần chỉ chọn một ảnh giả lập/không nhạy cảm.")
        return decode_image(next(iter(uploaded.values())))
    finally:
        for name in uploaded:
            created = Path(name).resolve()
            if created.parent == Path.cwd().resolve() and created not in before and created.is_file():
                created.unlink()
        uploaded.clear()

sample = Image.new("RGB", (32, 24), "white")
sample.info["comment"] = "must disappear"
sample_bytes = io.BytesIO()
sample.save(sample_bytes, format="PNG")
assert decode_image(sample_bytes.getvalue()).size == (32, 24)
assert not decode_image(sample_bytes.getvalue()).info
exif = Image.Exif()
exif[274] = 6
exif[271] = "Synthetic camera"
jpeg = io.BytesIO()
sample.save(jpeg, format="JPEG", exif=exif)
oriented = decode_image(jpeg.getvalue())
assert oriented.size == (24, 32) and not oriented.getexif() and not oriented.info
for invalid in (b"", b"not an image", b"x" * (MAX_UPLOAD_BYTES + 1)):
    try:
        decode_image(invalid)
    except (ValueError, OSError):
        pass
    else:
        raise AssertionError("Đã chấp nhận ảnh không hợp lệ.")
print("Tự kiểm đọc ảnh/giới hạn/xóa metadata: OK")

## 1. Nạp đúng một model

Dùng class official `Qwen3VLForEmbedding`, processor và cách lấy hidden state ở token cuối như
mã Qwen. Tiền xử lý bên dưới dừng khi ảnh lỗi; wrapper official có nhánh thay ảnh lỗi bằng `NULL`
nên không dùng wrapper đó. Mã Python tải về chỉ được import sau khi kiểm SHA-256.
`sdpa` và FP16 chạy trên T4; không cài FlashAttention. Gói phụ thuộc được ghi đầy đủ trong manifest.

In [ ]:
import importlib.util
import time
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, snapshot_download
from qwen_vl_utils import process_vision_info

MODEL_ID = "Qwen/Qwen3-VL-Embedding-2B"
MODEL_REVISION = "9f2f7e710d6d81056aa5c0a4f04764fec6bb7bda"
SOURCE_SHA256 = "8ffa74a1a6bb759610c57865ea416fd4daf9936cb787520e1112a3e1d547f36a"
INSTRUCTION = "Represent the user's input."
PROFILE = {
    "id": "findie-embedding-colab-v1", "instruction": INSTRUCTION,
    "dimension": 2048, "normalization": "L2", "pooling": "last_nonpadding_token",
    "dtype": "float16", "attention": "sdpa", "max_length": 2048,
    "min_pixels": 4 * 32 * 32, "max_pixels": 256 * 32 * 32,
    "image_patch_size": 16, "image_max_edge": MAX_IMAGE_EDGE,
    "batch_size": 1, "source_version": 1,
}
source_path = hf_hub_download(MODEL_ID, "scripts/qwen3_vl_embedding.py", revision=MODEL_REVISION)
if hashlib.sha256(Path(source_path).read_bytes()).hexdigest() != SOURCE_SHA256:
    raise RuntimeError("Mã inference không khớp SHA-256; dừng trước khi import.")
spec = importlib.util.spec_from_file_location("findie_qwen_embedding", source_path)
official = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = official
spec.loader.exec_module(official)
model_path = snapshot_download(
    MODEL_ID, revision=MODEL_REVISION, ignore_patterns=["*.md", ".gitattributes", "scripts/*"]
)
model = official.Qwen3VLForEmbedding.from_pretrained(
    model_path, torch_dtype=torch.float16, attn_implementation="sdpa",
    low_cpu_mem_usage=True, trust_remote_code=False,
).to("cuda").eval()
processor = official.Qwen3VLProcessor.from_pretrained(model_path, padding_side="right")
MANIFEST = {
    "experiment": "findie-embedding", "started_at": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "processor_revision": MODEL_REVISION, "inference_source_revision": MODEL_REVISION,
    "inference_source_sha256": SOURCE_SHA256, "profile": PROFILE,
    "profile_sha256": hashlib.sha256(json.dumps(PROFILE, sort_keys=True).encode()).hexdigest(),
    "environment": ENVIRONMENT,
}
print("Đã nạp", MODEL_ID, "revision", MODEL_REVISION)

In [ ]:
def validate_vector(vector):
    vector = np.asarray(vector, dtype=np.float32)
    if vector.shape != (2048,) or not np.isfinite(vector).all():
        raise ValueError("Vector phải có đúng 2048 phần tử hữu hạn.")
    norm = float(np.linalg.norm(vector))
    if not math.isfinite(norm) or norm <= 0:
        raise ValueError("Không chấp nhận vector không.")
    vector = vector / norm
    if not np.isclose(np.linalg.norm(vector), 1.0, atol=1e-5):
        raise ValueError("Chuẩn L2 không hợp lệ.")
    return vector

@torch.inference_mode()
def encode_one(*, text=None, image=None):
    if (text is None) == (image is None):
        raise ValueError("Chọn đúng một chế độ: text hoặc image.")
    content = []
    if text is not None:
        if not isinstance(text, str) or not text.strip() or len(text) > 2000:
            raise ValueError("Văn bản công khai phải có 1–2000 ký tự.")
        content.append({"type": "text", "text": text})
    else:
        if not isinstance(image, Image.Image):
            raise TypeError("Chỉ nhận ảnh PIL đã kiểm tra, không nhận URL.")
        content.append({"type": "image", "image": image,
                        "min_pixels": PROFILE["min_pixels"], "max_pixels": PROFILE["max_pixels"]})
    conversation = [
        {"role": "system", "content": [{"type": "text", "text": INSTRUCTION}]},
        {"role": "user", "content": content},
    ]
    prompt_text = processor.apply_chat_template([conversation], add_generation_prompt=True, tokenize=False)
    images, videos = process_vision_info([conversation], image_patch_size=16)
    if image is not None and not images:
        raise RuntimeError("Tiền xử lý không trả ảnh; không dùng vector thay thế.")
    inputs = processor(
        text=prompt_text, images=images, videos=videos,
        padding=True, do_resize=False, return_tensors="pt",
    )
    if inputs.input_ids.shape[1] > PROFILE["max_length"]:
        raise ValueError("Đầu vào vượt token budget; rút ngắn văn bản/ảnh rồi chạy lại.")
    inputs = inputs.to("cuda")
    output = model(**inputs, use_cache=False)
    last = inputs.attention_mask.shape[1] - 1 - inputs.attention_mask.flip(1).argmax(1)
    pooled = output.last_hidden_state[torch.arange(last.shape[0], device=last.device), last]
    vector = validate_vector(F.normalize(pooled.float(), p=2, dim=-1)[0].cpu().numpy())
    del output, pooled, inputs
    return vector

unit = validate_vector(np.ones(2048))
assert np.isclose(unit @ unit, 1, atol=1e-5)
for invalid in (np.zeros(2048), np.ones(3), np.full(2048, np.nan), np.full(2048, np.inf)):
    try:
        validate_vector(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("Đã nhận vector lỗi.")
print("Tự kiểm vector/cosine: OK")

## 2. Tiếng Việt → mô tả công khai

Thay `TEXT_QUERY` và các mô tả bằng dữ liệu demo nếu cần. Chưa so điểm với ảnh và chưa áp ngưỡng
chất lượng. Không thêm thông tin xác minh riêng vào những trường này.

In [ ]:
TEXT_QUERY = "Tôi làm mất ví màu xanh có khóa kéo."
TEXT_DOCUMENTS = [
    {"id": "vi-xanh", "text": "Nhặt được một chiếc ví màu xanh có khóa kéo tại khuôn viên trường."},
    {"id": "chia-khoa", "text": "Nhặt được chùm ba chìa khóa kim loại với móc màu đỏ."},
    {"id": "binh-nuoc", "text": "Bình nước giữ nhiệt màu trắng, có quai xách."},
]
records = []
text_query_vector = encode_one(text=TEXT_QUERY)
for document in TEXT_DOCUMENTS:
    vector = encode_one(text=document["text"])
    records.append({
        "id": document["id"], "modality": "text", "source_version": 1,
        "source_sha256": hashlib.sha256(document["text"].encode()).hexdigest(),
        "vector": vector.tolist(),
    })
text_ranking = sorted(
    ({"id": row["id"], "cosine": float(text_query_vector @ np.asarray(row["vector"], dtype=np.float32))}
     for row in records), key=lambda row: row["cosine"], reverse=True,
)
print(json.dumps(text_ranking, ensure_ascii=False, indent=2))

## 3. Ảnh → ảnh, chạy tuần tự từng ảnh

Mặc định tự vẽ ba đồ vật giả lập. Đặt `UPLOAD_IMAGE_QUERY = True` để upload **một ảnh không nhạy cảm**
làm truy vấn; muốn đánh giá thật cần bộ ảnh đã được duyệt và có nhãn phù hợp. Trong ứng dụng,
chỉ ảnh đã xử lý và duyệt đúng phiên bản mới được tạo vector công khai.

In [ ]:
UPLOAD_IMAGE_QUERY = False
image_documents = {}
for item_id, color, shape in [
    ("vi-xanh", "#2459a6", "wallet"),
    ("chia-khoa", "#a9a9a9", "key"),
    ("binh-nuoc", "#e8e8e8", "bottle"),
]:
    picture = Image.new("RGB", (384, 384), "#f4ead6")
    draw = ImageDraw.Draw(picture)
    if shape == "wallet":
        draw.rounded_rectangle((60, 110, 325, 270), radius=20, fill=color, outline="black", width=4)
        draw.line((80, 140, 300, 140), fill="white", width=5)
    elif shape == "key":
        draw.ellipse((80, 70, 210, 200), fill=color, outline="black", width=4)
        draw.rectangle((140, 170, 175, 315), fill=color, outline="black", width=4)
        draw.rectangle((170, 270, 210, 292), fill=color)
    else:
        draw.rounded_rectangle((130, 90, 250, 320), radius=18, fill=color, outline="black", width=4)
        draw.rectangle((145, 65, 235, 100), fill="#6c8183")
    image_documents[item_id] = picture
image_query = (upload_one_synthetic_image() if UPLOAD_IMAGE_QUERY else
               ImageOps.expand(image_documents["vi-xanh"].resize((320, 320)), border=32, fill="white"))
display(image_query)
image_query_vector = encode_one(image=image_query)
image_ranking = []
for item_id, picture in image_documents.items():
    vector = encode_one(image=picture)
    records.append({
        "id": item_id, "modality": "image", "source_version": 1,
        "source_sha256": image_sha256(picture), "width": picture.width, "height": picture.height,
        "review_status": "synthetic_demo_only", "vector": vector.tolist(),
    })
    image_ranking.append({"id": item_id, "cosine": float(image_query_vector @ vector)})
image_ranking.sort(key=lambda row: row["cosine"], reverse=True)
print(json.dumps(image_ranking, ensure_ascii=False, indent=2))

## 4. Xuất kết quả và giải phóng GPU

ZIP chỉ chứa vector/manifest/xếp hạng demo, không có ảnh gốc. Chưa tự nhập DB: .NET phải kiểm
revision, profile, kích thước, trạng thái duyệt và source version trước khi lưu. Nếu đổi prompt,
tiền xử lý, lượng tử hóa hoặc model, tạo profile mới và lập chỉ mục lại các vector liên quan.
Đặt `DOWNLOAD_RESULTS = True` để tải ZIP qua trình duyệt. Chạy cell dọn GPU trước notebook 02;
tốt nhất xóa runtime hiện tại rồi mở notebook 02 với runtime sạch.

In [ ]:
import zipfile

DOWNLOAD_RESULTS = False
MANIFEST["queries"] = {
    "text": {"source_sha256": hashlib.sha256(TEXT_QUERY.encode()).hexdigest(), "source_version": 1},
    "image": {"source_sha256": image_sha256(image_query), "source_version": 1,
              "width": image_query.width, "height": image_query.height},
}
export = {
    "manifest": MANIFEST, "vectors": records,
    "query_vectors": {"text": text_query_vector.tolist(), "image": image_query_vector.tolist()},
    "rankings": {"text": text_ranking, "image": image_ranking},
}
output = Path("/content/findie-embedding-results.zip")
with zipfile.ZipFile(output, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr("embedding-results.json", json.dumps(export, ensure_ascii=False, allow_nan=False, indent=2))
print("Kết quả:", output)
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(output))

In [ ]:
del model, processor, official
sys.modules.pop("findie_qwen_embedding", None)
gc.collect()
torch.cuda.empty_cache()
print("Đã giải phóng model; runtime vẫn giữ trọng số cache đến khi bị xóa.")